# 02 — Common preprocessing and figures

**Input:** the cleaned cohort from notebook 01  
**Does:** filters genera once using the full cohort and creates preprocessing/QC figures  
**Output:** one common filtered count table used by both methods, plus saved t-SNE/PCA coordinates and figures

Feature rules:
- present in at least 10% of samples
- at least 1% relative abundance in at least 10% of samples

The t-SNE and PCA are exploratory only. They use a CLR table made with a sample-specific pseudocount equal to half that sample's smallest positive retained value.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

PREVALENCE = 0.10
MIN_RELATIVE_ABUNDANCE = 0.01
SEED = 2026
COUNTRIES = ["FIN", "EST", "RUS"]
COLORS = {"FIN": "#0072B2", "EST": "#009E73", "RUS": "#D55E00"}

root = Path(".") if Path("data").exists() else Path("..")
input_folder = sorted((root / "data" / "processed_16s").iterdir())[-1]
output = root / "data" / "preprocessing" / datetime.now().strftime("%Y%m%d_%H%M%S")
figures = output / "figures"
figures.mkdir(parents=True)

counts = pd.read_csv(input_folder / "counts.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(input_folder / "metadata.csv", dtype={"sample_id": str, "subject_id": str, "country": str})
taxonomy = pd.read_csv(input_folder / "taxonomy.csv")
counts = counts.loc[metadata["sample_id"]]

print("Input:", input_folder)
print("Output:", output)

## Filter genera

The two filtering rules are written directly below so the retained feature set can be checked by hand.

In [ ]:
relative = counts.div(counts.sum(axis=1), axis=0)

prevalence = counts.gt(0).mean()
abundant = relative.ge(MIN_RELATIVE_ABUNDANCE).mean()

keep = (prevalence >= PREVALENCE) & (abundant >= PREVALENCE)
filtered = counts.loc[:, keep]

summary = pd.DataFrame({
    "measure": ["samples", "subjects", "genera_before", "genera_retained", "genera_removed"],
    "value": [
        len(metadata),
        metadata["subject_id"].nunique(),
        counts.shape[1],
        filtered.shape[1],
        counts.shape[1] - filtered.shape[1],
    ],
})

if filtered.shape[1] == 0:
    raise ValueError("No genera passed filtering.")

print(summary.to_string(index=False))

## CLR, PCA, and t-SNE

The CLR transformation is kept explicit. One set of t-SNE coordinates is saved and reused for both country and age plots.

In [ ]:
x = filtered.astype(float).copy()

for sample in x.index:
    positive = x.loc[sample, x.loc[sample] > 0]
    pseudocount = positive.min() / 2
    x.loc[sample] = x.loc[sample] + pseudocount

clr = np.log(x)
clr = clr.sub(clr.mean(axis=1), axis=0)

if not np.isfinite(clr.to_numpy()).all():
    raise ValueError("CLR contains non-finite values.")

pca_xy = PCA(n_components=2).fit_transform(clr)
tsne_xy = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED,
).fit_transform(clr)

coordinates = metadata[["sample_id", "subject_id", "age", "country"]].copy()
coordinates[["pca_1", "pca_2"]] = pca_xy
coordinates[["tsne_1", "tsne_2"]] = tsne_xy

In [ ]:
filtered.reset_index().to_csv(output / "counts_filtered.csv", index=False)
metadata.to_csv(output / "metadata.csv", index=False)
taxonomy[taxonomy["feature"].isin(filtered.columns)].to_csv(output / "taxonomy.csv", index=False)
pd.DataFrame({"feature": filtered.columns}).to_csv(output / "retained_features.csv", index=False)
coordinates.to_csv(output / "ordination_coordinates.csv", index=False)
summary.to_csv(output / "preprocessing_summary.csv", index=False)

feature_stats = pd.DataFrame({
    "feature": counts.columns,
    "prevalence": prevalence,
    "abundant_fraction": abundant,
    "retained": keep,
})
feature_stats.to_csv(output / "feature_filtering.csv", index=False)

## Preprocessing figures

In [ ]:
def save(name):
    plt.tight_layout()
    plt.savefig(figures / name, dpi=200)
    plt.show()
    plt.close()

for country in COUNTRIES:
    rows = coordinates["country"] == country
    plt.scatter(coordinates.loc[rows, "tsne_1"], coordinates.loc[rows, "tsne_2"],
                s=18, alpha=0.7, label=country, color=COLORS[country])
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("Samples by country")
plt.legend()
save("tsne_country.png")

plt.scatter(coordinates["tsne_1"], coordinates["tsne_2"],
            c=coordinates["age"], s=18, alpha=0.7, cmap="viridis")
plt.colorbar(label="Age at collection (days)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("Samples by age at collection")
save("tsne_age.png")

for country in COUNTRIES:
    rows = coordinates["country"] == country
    plt.scatter(coordinates.loc[rows, "pca_1"], coordinates.loc[rows, "pca_2"],
                s=18, alpha=0.7, label=country, color=COLORS[country])
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("CLR PCA by country")
plt.legend()
save("pca_country.png")

In [ ]:
metadata["country"].value_counts().reindex(COUNTRIES).plot.bar()
plt.ylabel("Samples")
plt.title("Samples by country")
save("samples_by_country.png")

metadata.drop_duplicates("subject_id")["country"].value_counts().reindex(COUNTRIES).plot.bar()
plt.ylabel("Subjects")
plt.title("Subjects by country")
save("subjects_by_country.png")

for country in COUNTRIES:
    metadata.loc[metadata["country"] == country, "age"].plot.hist(
        bins=20, alpha=0.45, label=country
    )
plt.xlabel("Age at collection (days)")
plt.title("Age at collection")
plt.legend()
save("age_distribution.png")

metadata.groupby("subject_id").size().plot.hist(bins=15)
plt.xlabel("Samples per subject")
plt.title("Longitudinal samples per subject")
save("samples_per_subject.png")

metadata["library_size"].plot.hist(bins=30)
plt.xlabel("Library size")
plt.title("Library size distribution")
save("library_size.png")

prevalence.plot.hist(bins=30)
plt.xlabel("Fraction of samples with count > 0")
plt.title("Genus prevalence")
save("genus_prevalence.png")

In [ ]:
summary.set_index("measure").loc[["genera_before", "genera_retained", "genera_removed"], "value"].plot.bar()
plt.ylabel("Genera")
plt.title("Feature filtering")
save("feature_filtering.png")

mean_abundance = relative[filtered.columns].mean().sort_values().tail(15)
labels = [name.split("|")[-1].replace("g__", "") for name in mean_abundance.index]
plt.barh(labels, mean_abundance.values)
plt.xlabel("Mean relative abundance")
plt.title("Most abundant retained genera")
save("top_genera.png")

print("Saved:", output)